In [ ]:
#@title 🚀 Bấm nút Play (▶) bên trái để bật VieNeu GPU Server
GPU_TIER = "Auto (Tự nhận T4 / L4 / A100)" #@param ["Auto (Tự nhận T4 / L4 / A100)", "T4 (Free/Pro - Batch 16)", "L4 (Colab Pro - Batch 32)", "A100 (Colab Pro+ - Batch 32)"]

import os, re, sys, time, json, shutil, subprocess, urllib.request
from IPython.display import HTML, clear_output, display

tier_map = {
    "Auto (Tự nhận T4 / L4 / A100)": "auto",
    "T4 (Free/Pro - Batch 16)": "T4",
    "L4 (Colab Pro - Batch 32)": "L4",
    "A100 (Colab Pro+ - Batch 32)": "A100",
}
tier = tier_map.get(GPU_TIER, "auto")

print("⏳ [1/4] Đang kiểm tra GPU & cài công cụ nền (espeak-ng, cloudflared)...")
subprocess.run("apt-get update -qq && apt-get install -y -qq espeak-ng > /dev/null 2>&1", shell=True, check=True)
if not shutil.which("cloudflared"):
    subprocess.run(
        "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True,
        check=True,
    )

repo_dir = "/content/vieneu-tts"
print("⏳ [2/4] Đang tải & cài đặt VieNeu-TTS (chỉ mất ~30s lần đầu)...")
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pnnbao97/VieNeu-TTS.git", repo_dir], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", repo_dir, "fastapi", "uvicorn", "huggingface_hub", "tokenizers"], check=True)

runner_path = os.path.join(repo_dir, "vieneu_colab_runner.py")
try:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/trankhiembaolong/quick-novel-translator/main/colab/vieneu_colab_runner.py",
        runner_path,
    )
except Exception as dl_err:
    print(f"⚠️ Không tải được runner mới ({dl_err}), dùng chế độ fallback mặc định.")
    runner_path = ""

print("⏳ [3/4] Đang khởi động VieNeu GPU Server & tải model AI (~1-2 phút lần đầu)...")
env = os.environ.copy()
env["HOST"] = "127.0.0.1"
env["PORT"] = "8000"
env["VIENEU_COLAB_GPU_TIER"] = tier
env["VIENEU_WATERMARK"] = "0"
env["PYTHONUNBUFFERED"] = "1"

server_log = open("/content/vieneu_server.log", "w", encoding="utf-8")
cmd = [sys.executable, runner_path] if (runner_path and os.path.exists(runner_path)) else [sys.executable, "-m", "apps.openai_speech"]
server_proc = subprocess.Popen(cmd, cwd=repo_dir, env=env, stdout=server_log, stderr=subprocess.STDOUT)

health_data = {}
deadline = time.time() + 600
while time.time() < deadline:
    if server_proc.poll() is not None:
        server_log.flush()
        raise RuntimeError("VieNeu server thoát sớm. Log:\n" + open("/content/vieneu_server.log", encoding="utf-8").read()[-2000:])
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as resp:
            if resp.status == 200:
                health_data = json.loads(resp.read().decode("utf-8"))
                break
    except Exception:
        pass
    time.sleep(2)

if not health_data:
    raise TimeoutError("VieNeu server khởi động quá 10 phút.")

print("⏳ [4/4] Đang mở Cloudflare HTTPS Tunnel miễn phí...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_url = ""
tunnel_deadline = time.time() + 60
while time.time() < tunnel_deadline:
    line = tunnel_proc.stdout.readline()
    if not line:
        if tunnel_proc.poll() is not None:
            break
        time.sleep(0.2)
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break

if not tunnel_url:
    raise RuntimeError("Không lấy được link trycloudflare.com từ cloudflared.")

gpu_label = health_data.get("gpu_name", "Colab GPU")
batch_label = health_data.get("batch_max_size", health_data.get("export_max_streams", 16))
dtype_label = health_data.get("model_dtype", "auto")
runtime_warn_html = ""
if tier in ("L4", "A100") and "T4" in gpu_label.upper():
    runtime_warn_html = f"<p style=\"margin: 8px 0 0; padding: 6px 10px; border-radius: 6px; background: #fef3c7; color: #92400e; font-size: 12px;\">💡 Bạn đang chọn <b>{tier}</b> nhưng Colab đang cấp phần cứng <b>{gpu_label}</b>. Để đổi lên L4/A100, bấm <b>Runtime → Change runtime type</b>.</p>"

clear_output(wait=True)
card_html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 620px; border: 2px solid #10b981; border-radius: 14px; padding: 18px 20px; background: linear-gradient(135deg, #ecfdf5 0%, #f0fdf4 100%); color: #064e3b;">
  <div style="display: flex; align-items: center; justify-content: space-between; gap: 10px; flex-wrap: wrap;">
    <div style="font-size: 16px; font-weight: 800;">🟢 VieNeu Colab GPU đã sẵn sàng!</div>
    <span style="background: #059669; color: #fff; padding: 4px 10px; border-radius: 999px; font-size: 12px; font-weight: 700;">{gpu_label} · Batch {batch_label} · {dtype_label}</span>
  </div>
  <p style="margin: 10px 0 12px; font-size: 13.5px; color: #065f46;">Bấm nút bên dưới để copy link, sau đó quay lại Extension Quick Novel Translator và bấm <b>📋 Dán</b> rồi <b>⬇ Tải Audio (WAV)</b>.</p>
  <div style="display: flex; gap: 8px; align-items: center; flex-wrap: wrap;">
    <input id="stv-colab-url" type="text" readonly value="{tunnel_url}" style="flex: 1; min-width: 240px; padding: 10px 12px; border: 1px solid #34d399; border-radius: 8px; font-size: 13px; font-weight: 600; background: #fff; color: #0f172a;" />
    <button id="stv-copy-btn" onclick="(function(){{var inp=document.getElementById('stv-colab-url');var btn=document.getElementById('stv-copy-btn');var done=function(){{btn.innerText='✓ Đã Copy! Quay lại Extension bấm Dán';btn.style.background='#047857';}};if(navigator.clipboard&&navigator.clipboard.writeText){{navigator.clipboard.writeText(inp.value).then(done).catch(function(){{inp.select();document.execCommand('copy');done();}});}}else{{inp.select();document.execCommand('copy');done();}}}})()" style="padding: 11px 18px; border: none; border-radius: 8px; background: #10b981; color: #fff; font-size: 14px; font-weight: 700; cursor: pointer;">📋 Copy Link Kết Nối</button>
  </div>
  {runtime_warn_html}
  <p style="margin: 10px 0 0; font-size: 12px; color: #047857;">⚠️ Giữ tab Colab này mở trong lúc Extension đang tải Audio.</p>
</div>
"""
display(HTML(card_html))

log_pos = os.path.getsize("/content/vieneu_server.log") if os.path.exists("/content/vieneu_server.log") else 0
try:
    while server_proc.poll() is None and tunnel_proc.poll() is None:
        if os.path.exists("/content/vieneu_server.log"):
            with open("/content/vieneu_server.log", "r", encoding="utf-8", errors="ignore") as lf:
                lf.seek(log_pos)
                chunk = lf.read()
                log_pos = lf.tell()
            for log_line in chunk.splitlines():
                if "BATCH done:" in log_line or "BATCH sent:" in log_line or "ERROR" in log_line:
                    print(log_line, flush=True)
        time.sleep(2)
except KeyboardInterrupt:
    pass
finally:
    for p in (tunnel_proc, server_proc):
        if p.poll() is None:
            p.terminate()
